# Reduced Dataset Composition
- In this notebook we present the process that we apply to compose a dataset with a reduced number of rows and columns, starting from the complete dataset (containing all the features and data point obtained after the merging and labelling steps). 
- We show the methodology we apply, not only for investigating the features, but also to prepare the data to be used by ML algorithms.
- The reduction of the rows that completes the composition of the reduced dataset is reported in the <strong>"NAME OF THE NOTEBOOK" </strong>
- We select the features by manual evaluation (delete malformed or irrelevant features, delete hidden labels), and using a features selection algorithm on the Tshark features to keep only the most useful for the detection.

## Dataset Processing and Evaluation

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from ast import literal_eval
# %pip install scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier

Note: you may need to restart the kernel to use updated packages.


In [4]:
# PATH = '/data/puccetti/space_data/final_merging_all_feature_ordered.csv'
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

input_dir = os.path.join(root_dir, "rospace_dataset", "3_complete_dataset")
output_dir = os.path.join(root_dir, "rospace_dataset", "4_reduced_and_noperiodicity_dataset")
os.makedirs(output_dir, exist_ok=True)

# 自動抓取 3_complete_dataset 裡最新的 cleaned_merged 檔案
search_pattern = os.path.join(input_dir, "cleaned_merged-*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    raise FileNotFoundError(f"找不到檔案，請確認 {input_dir} 中有資料！")

latest_csv = max(csv_files, key=os.path.getmtime)
PATH = latest_csv
date_val = os.path.basename(latest_csv).replace("cleaned_merged-", "").replace(".csv", "")

print(f"自動載入檔案: {PATH}")
print("-" * 40)

自動載入檔案: c:\Users\chuni\Desktop\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\3_complete_dataset\cleaned_merged-0531.csv
----------------------------------------


In [5]:
pd.set_option("display.max_columns", None)
pd.get_option("display.max_columns")

In [6]:
df = pd.read_csv(PATH, nrows=5000000, low_memory=False)

In [7]:
df = df.sort_values('timestamp')

In [8]:
# print(df['timestamp'])
print("Timestamp head/tail:\n", df['timestamp'])

Timestamp head/tail:
 0        1.780162e+09
1        1.780162e+09
2        1.780162e+09
3        1.780162e+09
4        1.780162e+09
             ...     
78169    1.780188e+09
78170    1.780188e+09
78171    1.780188e+09
78172    1.780188e+09
78173    1.780188e+09
Name: timestamp, Length: 78174, dtype: float64


In [9]:
# print(df.shape)
print("Shape:", df.shape)

Shape: (78174, 686)


## Some checks on columns and values

In [10]:
# print(df['attack'].value_counts())
print("Attack Value Counts:\n", df['attack'].value_counts())

Attack Value Counts:
 attack
observe                 58257
nmap port scanning      14659
nmap SYN flood           2577
metasploit SYN flood     2497
ros2 reconnaissance       136
nmap discovery             40
ros2 node crashing          6
ros2 reflection             2
Name: count, dtype: int64


Delete some unuseful columns: 
- 'Unnamed' columns are just duplicate indexes of dataframes

In [11]:
subs = "Unnamed"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [12]:
print(df.columns)

Index(['timestamp', 'layers.frame.frame.time', 'layers.frame.frame.time_utc',
       'layers.frame.frame.time_relative', 'layers.frame.frame.number',
       'layers.frame.frame.len', 'layers.frame.frame.cap_len',
       'layers.frame.frame.protocols', 'layers.ip.ip.dsfield',
       'layers.ip.ip.dsfield_tree.ip.dsfield.dscp',
       ...
       'Tcp_Close', 'nr_active_file', 'nr_inactive_file', 'topic_name',
       'src_topic', 'subscribers_count', 'publisher_count', 'msg_type',
       'msg_data', 'attack'],
      dtype='str', length=686)


## Delete Features related to time 
If we want to train an detector with a shuffled dataset (we want just to distinguish between normal and attack data points, without considering the chronological order of data point occurrences) we have to delete the features related to time as they mark attacks and normal behavoir and are not generalizable (hidden label).

However, we keep the timestamp to use it for time series analysis. The time stamp will be dropped based on the detector that we want to build.  

In [13]:
subs = "time"
res = [i for i in df.columns if subs in i and i != 'timestamp']
print(len(res))
print(res)
df=df.drop(res, axis=1)

11
['layers.frame.frame.time', 'layers.frame.frame.time_utc', 'layers.frame.frame.time_relative', 'layers.udp.Timestamps.udp.time_relative', 'layers.udp.Timestamps.udp.time_delta', 'layers.frame.frame.time_delta', 'layers.frame.frame.time_delta_displayed', 'layers.dns.dns.time', 'layers.icmp.tcp.Timestamps.tcp.time_relative', 'layers.icmp.tcp.Timestamps.tcp.time_delta', 'layers.icmp.ntp.ntp.reftime']


In [14]:
print(df.shape)

(78174, 675)


### Delete features that specify Source or Destination at diffrent layers of the protocol stack
The model generalization could be degradated by knowledge related to specific values observed during the monitoring campaign. The source and destination addresses are not generalizable, then, we drop them. 

In [15]:
subs = "dst"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

10
['layers.ip.ip.dst', 'layers.ip.ip.dst_host', 'layers.udp.udp.dstport', 'layers.udp.udp.dstport_tree._ws.expert._ws.expert.message', 'layers.udp.udp.dstport_tree._ws.expert._ws.expert.severity', 'layers.udp.udp.dstport_tree._ws.expert._ws.expert.group', 'layers.icmp.ip.ip.dst', 'layers.icmp.ip.ip.dst_host', 'layers.icmp.udp.udp.dstport', 'layers.icmp.tcp.tcp.dstport']


In [16]:
print(df.shape)

(78174, 665)


In [17]:
subs = "src"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

18
['layers.ip.ip.src', 'layers.ip.ip.src_host', 'layers.udp.udp.srcport', 'layers.rtps.rtps.guidPrefix.src', 'layers.rtps.rtps.guidPrefix.src_tree.rtps.hostId', 'layers.rtps.rtps.guidPrefix.src_tree.rtps.appId', 'layers.rtps.rtps.guidPrefix.src_tree.rtps.sm.guidPrefix.instanceId', 'layers.udp.udp.srcport_tree._ws.expert._ws.expert.message', 'layers.udp.udp.srcport_tree._ws.expert._ws.expert.severity', 'layers.udp.udp.srcport_tree._ws.expert._ws.expert.group', 'layers.icmp.ip.ip.src', 'layers.icmp.ip.ip.src_host', 'layers.icmp.udp.udp.srcport', 'layers.icmp.rtps.rtps.guidPrefix.src', 'layers.icmp.rtps.rtps.guidPrefix.src_tree.rtps.hostId', 'layers.icmp.rtps.rtps.guidPrefix.src_tree.rtps.appId', 'layers.icmp.rtps.rtps.guidPrefix.src_tree.rtps.sm.guidPrefix.instanceId', 'layers.icmp.tcp.tcp.srcport']


In [18]:
print(df.shape)

(78174, 647)


In [19]:
subs = "host"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

5
['layers.ip.ip.host', 'layers.rtps.rtps.sm.id_tree.serializedData.serializedData:.PID_PARTICIPANT_GUID.rtps.param.participant_guid_tree.rtps.param.guid.hostId', 'layers.icmp.ip.ip.host', 'layers.icmp.rtps.rtps.sm.id_tree.serializedData.serializedData:.PID_PARTICIPANT_GUID.rtps.param.participant_guid_tree.rtps.param.guid.hostId', 'layers.icmp.rtps.rtps.sm.id_tree.serializedData.serializedData:.PID_ENDPOINT_GUID.rtps.param.endpoint_guid_tree.rtps.param.guid.hostId']


In [20]:
print(df.shape)

(78174, 642)


In [21]:
subs = "addr"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

2
['layers.ip.ip.addr', 'layers.icmp.ip.ip.addr']


In [22]:
print(df.shape)

(78174, 640)


### Delete features related to the "Frame" protocol
From wireshark doc (https://wiki.wireshark.org/Protocols/frame):

"The frame protocol isn't a real protocol itself, but used by Wireshark as a base for all the protocols on top of it. It shows information from capturing, such as the exact time a specific frame was captured. You could think of it as a pseudo dissector."

In [23]:
subs = ".frame."
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

4
['layers.frame.frame.number', 'layers.frame.frame.len', 'layers.frame.frame.cap_len', 'layers.frame.frame.protocols']


In [24]:
print(df.shape)

(78174, 636)


### Delete features that contains ID keyword
We want the dataset to be more general as possible. We drop the ID wich are specific to the execution of the system during the monitoring campaign. Also, the ID can implicitly be an hidden label. For example, the attacker can be associated, during the training to a specific id. However, at test time the association can be different, degrading the performance of the model.

First, the features are printed to ensure that we do not drop features with substring "id" in the name that are relevant.

In [25]:
subs = "id"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

352
['layers.ip.ip.id', 'layers.rtps.rtps.guidPrefix', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.domain_id', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.participant_idx', 'layers.rtps.rtps.sm.id', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.reserved', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.data.serialized_key', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.data_present', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.inline_qos', 'layers.rtps.rtps.sm.id_tree.rtps.sm.flags_tree.rtps.flag.endianness', 'layers.rtps.rtps.sm.id_tree.rtps.sm.octetsToNextHeader', 'layers.rtps.rtps.sm.id_tree.rtps.extra_flags', 'layers.rtps.rtps.sm.id_tree.rtps.octets_to_inline_qos', 'layers.rtps.rt

In [26]:
print(df.shape)

(78174, 284)


In [27]:
subs = "port"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

35
['layers.udp.udp.port', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.rtps.Default port mapping: domainId=Unknown, participantIdx=120, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=1, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.icmp.udp.udp.port', 'layers.icmp.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=1, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.icmp.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_METATRAFFIC.rtps.traffic_nature', 'layers.icmp.rtps.Default port mapping (Based on calculated domainId. Might not be accurate): domainId=0, participantIdx=0, nature=UNICAST_US

In [28]:
print(df.shape)

(78174, 249)


### Delete malformed features
These features seems to be badly formatted and can be the results of a formatting exeption during the captures. 

In [29]:
subs = "ubuntu"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

20
['layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.name', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.name.len', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.count.labels', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.type', 'layers.dns.Queries.connectivity-check.ubuntu.com: type A, class IN.dns.qry.class', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.name', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.name.len', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.count.labels', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.type', 'layers.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.class', 'layers.icmp.dns.Queries.connectivity-check.ubuntu.com: type AAAA, class IN.dns.qry.name', 'layers.icmp.dns.Queries.connectivity-chec

In [30]:
print(df.shape)

(78174, 229)


In [31]:
subs = "microsoft"
res = [i for i in df.columns if subs in i and i != 'src_topic']
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [32]:
print(df.shape)

(78174, 229)


### Manual evaluation of the Tshark features 

In [33]:
subs = "."
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
#df=df.drop(res, axis=1)

196
['layers.ip.ip.dsfield', 'layers.ip.ip.dsfield_tree.ip.dsfield.dscp', 'layers.ip.ip.len', 'layers.ip.ip.flags', 'layers.ip.ip.flags_tree.ip.flags.df', 'layers.ip.ip.ttl', 'layers.ip.ip.proto', 'layers.ip.ip.checksum', 'layers.ip.ip.stream', 'layers.udp.udp.length', 'layers.udp.udp.checksum', 'layers.udp.udp.checksum.status', 'layers.udp.udp.stream', 'layers.udp.udp.stream.pnum', 'layers.udp.udp.payload', 'layers.rtps.rtps.magic', 'layers.rtps.rtps.version', 'layers.rtps.rtps.version_tree.rtps.version.major', 'layers.rtps.rtps.version_tree.rtps.version.minor', 'layers.rtps.rtps.vendorId', 'layers.dns.dns.flags', 'layers.dns.dns.flags_tree.dns.flags.response', 'layers.dns.dns.flags_tree.dns.flags.opcode', 'layers.dns.dns.flags_tree.dns.flags.truncated', 'layers.dns.dns.flags_tree.dns.flags.recdesired', 'layers.dns.dns.flags_tree.dns.flags.z', 'layers.dns.dns.flags_tree.dns.flags.ad', 'layers.dns.dns.flags_tree.dns.flags.checkdisable', 'layers.dns.dns.count.queries', 'layers.dns.dns.c

### Drop the malformed features 
We drop the features with "/" or "\"

In [34]:
subs = "/"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [35]:
subs = "\\"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [36]:
print(df.shape)

(78174, 229)


In [37]:
subs = "PTR"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

12
['layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.name', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.name.len', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.count.labels', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.type', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.class', 'layers.mdns.Queries._ipp._tcp.local: type PTR, class IN, "QM" question.dns.qry.qu', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.name', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.name.len', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.count.labels', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.type', 'layers.mdns.Queries._ipps._tcp.local: type PTR, class IN, "QM" question.dns.qry.class', 'layers.mdns.Queri

In [38]:
print(df.shape)

(78174, 217)


In [39]:
subs = "full_uri"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [40]:
subs = "request_number"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


In [41]:
subs = "<Root>"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

18
['layers.dns.Additional records.<Root>: type OPT.dns.resp.name', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.type', 'layers.dns.Additional records.<Root>: type OPT.dns.rr.udp_payload_size', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.ext_rcode', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.edns0_version', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.z', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.z_tree.dns.resp.z.do', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.z_tree.dns.resp.z.reserved', 'layers.dns.Additional records.<Root>: type OPT.dns.resp.len', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.name', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.type', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.rr.udp_payload_size', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.ext_rcode', 'layers.icmp.dns.Additional records.<Root>: type OPT.dns.resp.edns0_ver

In [42]:
print(df.shape)

(78174, 199)


In [43]:
subs = "len"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

11
['layers.ip.ip.len', 'layers.udp.udp.length', 'layers.dns.Queries.cdn.fwupd.org: type A, class IN.dns.qry.name.len', 'layers.dns.Queries.cdn.fwupd.org: type AAAA, class IN.dns.qry.name.len', 'layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type A, class IN.dns.qry.name.len', 'layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type AAAA, class IN.dns.qry.name.len', 'layers.icmp.ip.ip.hdr_len', 'layers.icmp.ip.ip.len', 'layers.icmp.udp.udp.length', 'layers.icmp.tcp.tcp.hdr_len', 'layers.icmp.ssh.ssh.packet_length_encrypted']


In [44]:
print(df.shape)

(78174, 188)


In [45]:
subs = "seq"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

1
['layers.icmp.tcp.tcp.seq']


### Save list of features 

In [46]:
features = df.columns

In [47]:
dict = {'features': features}
     
df_features = pd.DataFrame(dict)

In [48]:
# df_features.to_csv("/data/puccetti/space_data/features_usable_temp.csv")
feat_temp_path = os.path.join(output_dir, f"features_usable_temp_{date_val}.csv")
df_features.to_csv(feat_temp_path, index=False)

### Create dataset with the subset of the features 
The objecftive is to understand the memory occupation of the resulting dataset

In [49]:
# PATH = '/data/puccetti/space_data/final_merging_all_feature_ordered.csv'

In [50]:
# features = pd.read_csv("/data/puccetti/space_data/features_usable_temp.csv")
features = pd.read_csv(feat_temp_path)
to_load = features['features'].values.tolist()

In [51]:
df = pd.read_csv(PATH, usecols=to_load, low_memory=False)

In [52]:
# print(df.shape)
print("Shape after reloading:", df.shape)

Shape after reloading: (78174, 187)


In [53]:
# df.to_csv("/data/puccetti/space_data/usable_temp.csv")
usable_temp_path = os.path.join(output_dir, f"usable_temp_{date_val}.csv")
df.to_csv(usable_temp_path, index=False)

In [54]:
# df = pd.read_csv("/data/puccetti/space_data/usable_temp.csv")
df = pd.read_csv(usable_temp_path, low_memory=False)

# Prepare the data for training
In this section, we prepare the data to be processed by ML algorithms. In particular, we perform the following steps:
- <strong>Convert mixed dtypes </strong>: we uniform the type of features with mixed type values.
- <strong>Handle NaN values</strong>: we replace NaN and infinite values with -1. 
- <strong>Convert Label to Numeric</strong>: We substitute label values with numeric values. Then, we create two versions of the dataset: with  binary labels (attack, normal), and with multiple labels (one label for each attack).
- <strong>Convert String To Numeric</strong>: we convert the string values to numbers using categorical encoding. This technique assigns a unique number to any unique string values of a feature.
- <strong>Split the dataset in Training and Test sets</strong>: after removing labels and timestamps columns, we split the dataframe in training and test sets with a 60/40 split.

## Convert mixed dtypes
We convert mixed dtypes columns to string using the following lambda function:

In [55]:
def convert_dtype(x):
    # if not x:
    if pd.isna(x) or x == '':
        return ''
    try:
        return str(x)   
    except:        
        return ''
    
def convert_hex(x):
    # if not x:
    if pd.isna(x) or x == '':
        return 0
    try:
        return literal_eval(x)
    except:        
        return 0

In [56]:
#Indexes of the columns to be converted
# to_convert = [7,10,17,21,31,35,38,41,42,45,49,53,62,64,65,66,69,81,82,85,86,87,90,91,94,96,103,106,111,113,116,120,122,124,127,134,136,139,162,173,176,178,181,184,190,195,197,198,199,225,228,229]
to_convert_cols = df.select_dtypes(include=['object', 'str']).columns

In [57]:
#Convert
# for i in to_convert:
#     df[df.columns[i]] = df[df.columns[i]].apply(lambda x: convert_dtype(x))
for col in to_convert_cols:
    if col != 'attack':
        df[col] = df[col].apply(lambda x: convert_dtype(x))

In [58]:
# print(df.shape)
print("Shape after dtype conversion:", df.shape)

Shape after dtype conversion: (78174, 187)


### Handling NaN values

In [59]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())

In [60]:
print(nanv)

[np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.True_, np.True_, np.True_, np.False_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.False_, np.False_, np.True_, np.True_, np.False_, np.True_, np.False_, np.True_, np.True_, np.True_, np.False_, np.True_, np.False_, np.True_, np.False_, np.True_, np.True_, np.False_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.False_, np.True_, np.True_, np.False_, np.True_, np.True_, np.False_, np.False_, np.False_, np.True_, np.True_, np.F

In [61]:
df.replace([np.inf, -np.inf], -1, inplace=True)
df.fillna(-1, inplace=True)
df=df.dropna(thresh=1, axis=1)

In [62]:
df.replace('nan', -1, inplace=True)

,timestamp,layers.ip.ip.dsfield,layers.ip.ip.dsfield_tree.ip.dsfield.dscp,layers.ip.ip.flags,layers.ip.ip.flags_tree.ip.flags.df,layers.ip.ip.ttl,layers.ip.ip.proto,layers.ip.ip.checksum,layers.ip.ip.stream,layers.udp.udp.checksum,layers.udp.udp.checksum.status,layers.udp.udp.stream,layers.udp.udp.stream.pnum,layers.udp.udp.payload,layers.rtps.rtps.magic,layers.rtps.rtps.version,layers.rtps.rtps.version_tree.rtps.version.major,layers.rtps.rtps.version_tree.rtps.version.minor,layers.rtps.rtps.vendorId,layers.dns.dns.flags,layers.dns.dns.flags_tree.dns.flags.response,layers.dns.dns.flags_tree.dns.flags.opcode,layers.dns.dns.flags_tree.dns.flags.truncated,layers.dns.dns.flags_tree.dns.flags.recdesired,layers.dns.dns.flags_tree.dns.flags.z,layers.dns.dns.flags_tree.dns.flags.ad,layers.dns.dns.flags_tree.dns.flags.checkdisable,layers.dns.dns.count.queries,layers.dns.dns.count.answers,layers.dns.dns.count.auth_rr,layers.dns.dns.count.add_rr,layers.ip.ip.ttl_tree._ws.expert._ws.expert.message,layers.ip.ip.ttl_tree._ws.expert._ws.expert.severity,layers.ip.ip.ttl_tree._ws.expert._ws.expert.group,layers.dns.dns.flags_tree.dns.flags.authoritative,layers.dns.dns.flags_tree.dns.flags.recavail,layers.dns.dns.flags_tree.dns.flags.authenticated,layers.dns.dns.flags_tree.dns.flags.rcode,layers.dns.dns.response_to,layers.mdns.dns.flags,layers.mdns.dns.flags_tree.dns.flags.response,layers.mdns.dns.flags_tree.dns.flags.opcode,layers.mdns.dns.flags_tree.dns.flags.truncated,layers.mdns.dns.flags_tree.dns.flags.recdesired,layers.mdns.dns.flags_tree.dns.flags.z,layers.mdns.dns.flags_tree.dns.flags.checkdisable,layers.mdns.dns.count.queries,layers.mdns.dns.count.answers,layers.mdns.dns.count.auth_rr,layers.mdns.dns.count.add_rr,"layers.dns.Queries.cdn.fwupd.org: type A, class IN.dns.qry.name","layers.dns.Queries.cdn.fwupd.org: type A, class IN.dns.count.labels","layers.dns.Queries.cdn.fwupd.org: type A, class IN.dns.qry.type","layers.dns.Queries.cdn.fwupd.org: type A, class IN.dns.qry.class","layers.dns.Queries.cdn.fwupd.org: type AAAA, class IN.dns.qry.name","layers.dns.Queries.cdn.fwupd.org: type AAAA, class IN.dns.count.labels","layers.dns.Queries.cdn.fwupd.org: type AAAA, class IN.dns.qry.type","layers.dns.Queries.cdn.fwupd.org: type AAAA, class IN.dns.qry.class","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type A, class IN.dns.qry.name","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type A, class IN.dns.count.labels","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type A, class IN.dns.qry.type","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type A, class IN.dns.qry.class","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type AAAA, class IN.dns.qry.name","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type AAAA, class IN.dns.count.labels","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type AAAA, class IN.dns.qry.type","layers.dns.Queries.cdn.fwupd.org.dlinkrouter.baf6.local: type AAAA, class IN.dns.qry.class",layers.icmp.icmp.type,layers.icmp.icmp.type_tree._ws.expert._ws.expert.message,layers.icmp.icmp.type_tree._ws.expert._ws.expert.severity,layers.icmp.icmp.type_tree._ws.expert._ws.expert.group,layers.icmp.icmp.code,layers.icmp.icmp.checksum,layers.icmp.icmp.checksum.status,layers.icmp.icmp.unused,layers.icmp.ip.ip.version,layers.icmp.ip.ip.dsfield,layers.icmp.ip.ip.dsfield_tree.ip.dsfield.dscp,layers.icmp.ip.ip.dsfield_tree.ip.dsfield.ecn,layers.icmp.ip.ip.flags,layers.icmp.ip.ip.flags_tree.ip.flags.rb,layers.icmp.ip.ip.flags_tree.ip.flags.df,layers.icmp.ip.ip.flags_tree.ip.flags.mf,layers.icmp.ip.ip.frag_offset,layers.icmp.ip.ip.ttl,layers.icmp.ip.ip.proto,layers.icmp.ip.ip.checksum,layers.icmp.ip.ip.checksum.status,layers.icmp.ip.ip.stream,layers.icmp.udp.udp.checksum,layers.icmp.udp.udp.checksum.status,layers.icmp.udp.udp.stream,layers.icmp.udp.udp.payload,layers.icmp.rtps.rtps.magic,layers.icmp.rtps.rtps.version,layers.icmp.rtps.rtps.version_tree.rtps.version.major,

In [63]:
nanv = []
for col in df.columns:
    nanv.append(df[col].isnull().values.any())
print(nanv)

[np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_

In [64]:
#Save the processed dataset
# df.to_csv('/data/puccetti/space_data/usable_temp_nan.csv')
#df = pd.read_csv('/data/puccetti/space_data/final_full_dataset_nan.csv')
usable_temp_nan_path = os.path.join(output_dir, f"usable_temp_nan_{date_val}.csv")
df.to_csv(usable_temp_nan_path, index=False)

### Convert Label columns to Numeric values
We substitute label values with numeric values. We create two version of the dataset:
- binary classification
- multiple label classification

In [65]:
print(df['timestamp'])

0        1.780162e+09
1        1.780162e+09
2        1.780162e+09
3        1.780162e+09
4        1.780162e+09
             ...     
78169    1.780188e+09
78170    1.780188e+09
78171    1.780188e+09
78172    1.780188e+09
78173    1.780188e+09
Name: timestamp, Length: 78174, dtype: float64


In [66]:
# print(df['attack'].value_counts())
print("Attack counts before mapping:\n", df['attack'].value_counts())



Attack counts before mapping:
 attack
observe                 58257
nmap port scanning      14659
nmap SYN flood           2577
metasploit SYN flood     2497
ros2 reconnaissance       136
nmap discovery             40
ros2 node crashing          6
ros2 reflection             2
Name: count, dtype: int64


In [67]:
df['attack'] = df['attack'].replace('metasploit SYN flood', 1) 
df['attack'] = df['attack'].replace('nmap discovery', 2)
df['attack'] = df['attack'].replace('nmap SYN flood', 3) 
df['attack'] = df['attack'].replace('ros2 node crashing', 4)
df['attack'] = df['attack'].replace('ros2 reconnaissance', 5)
df['attack'] = df['attack'].replace('ros2 reflection', 6)
df['attack'] = df['attack'].replace('nmap port scanning', 7)
df['attack'] = df['attack'].replace('observe', 0)

# df['attack'] = pd.to_numeric(df['attack'])
df['attack'] = pd.to_numeric(df['attack'], errors='coerce').fillna(0).astype(int)

df['attack'].unique(), df['attack'].nunique()

(array([2, 0, 7, 5, 3, 6, 4, 1]), 8)

In [68]:
# print(df['attack'].value_counts())
print("Attack counts after multi-mapping:\n", df['attack'].value_counts())

Attack counts after multi-mapping:
 attack
0    58257
7    14659
3     2577
1     2497
5      136
2       40
4        6
6        2
Name: count, dtype: int64


In [69]:
# df.to_csv('/data/puccetti/space_data/usable_temp_multi.csv')
usable_temp_multi_path = os.path.join(output_dir, f"usable_temp_multi_{date_val}.csv")
df.to_csv(usable_temp_multi_path, index=False)

In [70]:
df['attack'] = df['attack'].replace(2, 1)
df['attack'] = df['attack'].replace(3, 1) 
df['attack'] = df['attack'].replace(4, 1)
df['attack'] = df['attack'].replace(5, 1)
df['attack'] = df['attack'].replace(6, 1)
df['attack'] = df['attack'].replace(7, 1)

In [71]:
# print(df['attack'].value_counts())
print("Attack counts after binary-mapping:\n", df['attack'].value_counts())

Attack counts after binary-mapping:
 attack
0    58257
1    19917
Name: count, dtype: int64


In [72]:
# df.to_csv('/data/puccetti/space_data/usable_temp_bin.csv')
usable_temp_bin_path = os.path.join(output_dir, f"usable_temp_bin_{date_val}.csv")
df.to_csv(usable_temp_bin_path, index=False)

## Convert String to Numeric

In [73]:
list_column_string=df.select_dtypes(exclude=[np.number]).columns

for i in list_column_string:
    if i != 'timestamp':
        df[i] = pd.Categorical(df[i])

In [74]:
for i in list_column_string:
    if i != 'timestamp':
        df[i] = df[i].cat.codes

### Split the dataset to create Train and Test Sets

In [75]:
from sklearn.model_selection import train_test_split

In [76]:
df_clean = df.copy()

In [77]:
#df = df.drop(['Unnamed: 0'], axis=1)
df = df.drop(['timestamp'], axis=1)
print("Dataset shape before ExtraTrees: " + str(df.shape))

Dataset shape before ExtraTrees: (78174, 186)


In [78]:
subs = "Unnamed"
res = [i for i in df.columns if subs in i]
print(len(res))
print(res)
df=df.drop(res, axis=1)

0
[]


I want to make the feature selection only on the feature related to the network monitor (Tshark).

In [79]:
subs = "."
res = [i for i in df.columns if subs in i]
print(len(res))

154


In [80]:
label = df['attack']
df = df.drop(['attack'], axis=1)

x_train, x_test, y_train, y_test = train_test_split(df[res], label, test_size=0.4, random_state=42)

x_train = x_train.to_numpy()
x_test = x_test.to_numpy()

In [81]:
print("Train Set Shape: " + str(x_train.shape))
print("Train Set Label Shape: " + str(y_train.shape))
print("Test Set Shape: " + str(x_test.shape))
print("Test Set Label Shape: " + str(y_test.shape))

Train Set Shape: (46904, 154)
Train Set Label Shape: (46904,)
Test Set Shape: (31270, 154)
Test Set Label Shape: (31270,)


# Select best features for the light version of the dataset
We use the feature ranking algorithm of ExtraTreesClassifier to select the best Tshark features. 

In [82]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectFromModel

In [83]:
# clf = ExtraTreesClassifier(n_estimators=30)
clf = ExtraTreesClassifier(n_estimators=30, random_state=42)
clf = clf.fit(x_train, y_train)
clf.feature_importances_

array([8.66292832e-05, 2.83483125e-04, 0.00000000e+00, 0.00000000e+00,
       2.83999291e-02, 2.46368190e-03, 1.17446606e-01, 2.72104865e-02,
       1.03760221e-01, 1.74478264e-04, 3.01373536e-02, 5.23276304e-01,
       3.55171668e-02, 8.57377940e-05, 4.72084499e-06, 0.00000000e+00,
       2.90678765e-03, 1.16188635e-02, 4.44252719e-03, 3.16372367e-03,
       0.00000000e+00, 1.27074812e-02, 9.83988198e-03, 3.39091575e-03,
       1.99158241e-03, 3.10979927e-03, 1.58187073e-02, 9.31656547e-03,
       0.00000000e+00, 9.31656547e-03, 6.25631295e-04, 4.75209377e-04,
       7.20853405e-08, 1.96725097e-03, 1.65279725e-07, 3.02606270e-04,
       5.59750606e-04, 9.02024830e-03, 4.74700356e-07, 7.07927607e-06,
       1.13606387e-05, 4.71915633e-06, 1.22287357e-05, 1.65019537e-06,
       2.35957817e-06, 2.07783653e-06, 6.66541123e-06, 2.15188485e-05,
       0.00000000e+00, 5.12531162e-05, 8.13086133e-05, 1.20238554e-04,
       7.13585096e-05, 4.57311902e-05, 2.42880725e-05, 3.04636913e-05,
      

In [84]:
importances = clf.feature_importances_
indices = np.argsort(importances)[-30:]

In [85]:
print(indices)

[31 36 30 89 33 24  5 70 16 25 19 23 18 90 87 37 29 27 22 17 21 84 26  7
  4 10 12  8  6 11]


In [86]:
best_features = df[res].columns[indices]

In [87]:
subs = "."
# res = [i for i in df.columns if subs not in i]
res_no_dot = [i for i in df.columns if subs not in i]
print(len(res_no_dot))

31


In [88]:
best_features = set(best_features)
# res = set(res)
res_no_dot = set(res_no_dot)
union = list(best_features.union(res_no_dot))

In [89]:
print(union)
print(len(union))

['nr_active_file', 'msg_type', 'msg_data', 'layers.icmp.udp.udp.stream', 'pgfree', 'layers.dns.dns.flags_tree.dns.flags.recdesired', 'layers.ip.ip.stream', 'Tcp_Established', 'Net_Received', 'Tcp_Syn', 'pgpgin', 'pgpgout', 'pgfault', 'Tcp_Close', 'Buffers', 'pgalloc_dma', 'layers.icmp.udp.udp.checksum', 'layers.ip.ip.proto', 'layers.rtps.rtps.version_tree.rtps.version.minor', 'layers.dns.dns.count.queries', 'layers.dns.dns.count.answers', 'Net_Sent', 'layers.icmp.udp.udp.payload', 'Inactive', 'Cached', 'layers.ip.ip.ttl_tree._ws.expert._ws.expert.severity', 'layers.ip.ip.checksum', 'src_topic', 'Active', 'layers.dns.dns.flags_tree.dns.flags.truncated', 'layers.dns.dns.flags_tree.dns.flags.checkdisable', 'layers.dns.dns.flags', 'layers.icmp.icmp.checksum', 'SwapFree', 'layers.ip.ip.ttl', 'pgactivate', 'layers.ip.ip.ttl_tree._ws.expert._ws.expert.message', 'layers.dns.dns.flags_tree.dns.flags.rcode', 'layers.dns.dns.flags_tree.dns.flags.z', 'layers.udp.udp.stream', 'Disk_Read', 'Tcp_Time

In [90]:
# np.save('/data/puccetti/space_data/usable_features.npy', union, allow_pickle=True)
usable_features_npy_path = os.path.join(output_dir, f"usable_features_{date_val}.npy")
np.save(usable_features_npy_path, union, allow_pickle=True)

# Compose the reduced dataset

In [91]:
# features = np.load('/data/puccetti/space_data/usable_features.npy')
features = np.load(usable_features_npy_path, allow_pickle=True)

In [92]:
features = list(features)

In [93]:
features.append('attack')

In [94]:
if 'timestamp' in df_clean.columns:
    features.append('timestamp')

In [95]:
# df = pd.read_csv(PATH, usecols=features, low_memory=False)
features_to_keep = [c for c in features if c in df_clean.columns]
df_final = df_clean[features_to_keep]

In [96]:
# print(df['attack'])
print("Final Attack counts:\n", df_final['attack'].value_counts())

Final Attack counts:
 attack
0    58257
1    19917
Name: count, dtype: int64


In [97]:
# df.to_csv('/data/puccetti/space_data/reduced_final.csv')
reduced_final_path = os.path.join(output_dir, f"reduced_final_{date_val}.csv")
df_final.to_csv(reduced_final_path, index=False)
print(f"\n✅ 成功！降維後的最終資料已存至: {reduced_final_path}")


✅ 成功！降維後的最終資料已存至: c:\Users\chuni\Desktop\Zero-Trust-Authentication-Based-on-ROS2\rospace_dataset\4_reduced_and_noperiodicity_dataset\reduced_final_0531.csv


In [98]:
#df = pd.read_csv('/data/puccetti/space_data/usable_temp_bin.csv', usecols=features)

In [99]:
#df.to_csv('/data/puccetti/space_data/usable_final_bin.csv')

In [100]:
#df = pd.read_csv('/data/puccetti/space_data/usable_temp_multi.csv', usecols=features)

In [101]:
#df.to_csv('/data/puccetti/space_data/usable_final_multi.csv')

In [102]:
import pandas as pd
import numpy as np
import glob
import os

# ==========================================
# 1. 自動定位路徑與最新檔案
# ==========================================
current_dir = os.getcwd()
project_name = "Zero-Trust-Authentication-Based-on-ROS2"
if project_name in current_dir:
    root_dir = current_dir[:current_dir.find(project_name) + len(project_name)]
else:
    root_dir = current_dir

# 定位到第四步的資料夾
check_dir = os.path.join(root_dir, "rospace_dataset", "4_reduced_and_noperiodicity_dataset")

# 自動搜尋最新的 reduced_final 檔案
search_pattern = os.path.join(check_dir, "reduced_final_*.csv")
csv_files = glob.glob(search_pattern)

if not csv_files:
    print(f"❌ 錯誤：在 {check_dir} 找不到任何 reduced_final 檔案！")
else:
    # 抓取最後修改的檔案
    latest_csv = max(csv_files, key=os.path.getmtime)
    print(f"🚀 [啟動檢查] 檔案名稱: {os.path.basename(latest_csv)}")
    print("-" * 50)

    # 讀取檔案
    df = pd.read_csv(latest_csv, low_memory=False)

    # --- 檢查 1：檔案維度 ---
    print(f"📊 [1. 維度檢查]")
    print(f"   - 資料筆數 (Rows): {df.shape[0]}")
    print(f"   - 特徵數量 (Columns): {df.shape[1]}")
    if 30 <= df.shape[1] <= 70:
        print("   🟢 狀態: 欄位數量合理（包含 Top 30 網路特徵 + 系統指標）。")
    else:
        print("   ⚠️ 提醒: 欄位數量較多，請確認是否包含過多重複欄位。")

    # --- 檢查 2：幽靈欄位 (Unnamed) ---
    print(f"\n👻 [2. 幽靈欄位檢查]")
    unnamed_cols = [c for c in df.columns if 'Unnamed' in c]
    if len(unnamed_cols) == 0:
        print("   🟢 狀態: 完美！沒有殘留任何 Unnamed 欄位。")
    else:
        print(f"   🔴 錯誤: 發現 {len(unnamed_cols)} 個幽靈欄位: {unnamed_cols}")

    # --- 檢查 3：攻擊標籤 (Label) 純淨度 ---
    print(f"\n🎯 [3. 攻擊標籤檢查]")
    if 'attack' in df.columns:
        counts = df['attack'].value_counts(dropna=False).to_dict()
        print(f"   - 標籤分佈: {counts}")
        
        # 檢查是否只含有 0 和 1 (且必須是數字型態)
        unique_vals = set(df['attack'].unique())
        is_numeric = np.issubdtype(df['attack'].dtype, np.number)
        
        if unique_vals.issubset({0, 1}) and is_numeric:
            print("   🟢 狀態: 完美！標籤已成功轉為數字 0 (正常) 與 1 (攻擊)。")
        else:
            print(f"   🔴 錯誤: 標籤異常！含有非預期數值 {unique_vals} 或型態為 {df['attack'].dtype}")
    else:
        print("   🔴 致命錯誤: 找不到 'attack' 欄位！")

    # --- 檢查 4：遺失值 (NaN) 與 型態 ---
    print(f"\n🧼 [4. 資料品質檢查]")
    nan_count = df.isnull().sum().sum()
    if nan_count == 0:
        print("   🟢 狀態: 完美！全檔案無遺失值 (NaN)。")
    else:
        print(f"   🔴 錯誤: 檔案中還有 {nan_count} 個 NaN！這會導致訓練失敗。")

    # 檢查是否還有字串型態 (除了 timestamp 以外)
    string_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
    # 移除 timestamp，因為它是我們允許的唯一數字字串(或是浮點數)
    if 'timestamp' in string_cols: string_cols.remove('timestamp')
    
    if len(string_cols) == 0:
        print("   🟢 狀態: 完美！所有特徵欄位皆為數值型態。")
    else:
        print(f"   🔴 錯誤: 發現字串欄位: {string_cols} (請在清洗腳本中處理)。")

    print("-" * 50)
    print("✅ 檢查完成！如果上方全是綠燈，你可以放心進入下一步：週期性處理 (No-Periodicity)。")

🚀 [啟動檢查] 檔案名稱: reduced_final_0531.csv
--------------------------------------------------
📊 [1. 維度檢查]
   - 資料筆數 (Rows): 78174
   - 特徵數量 (Columns): 63
   🟢 狀態: 欄位數量合理（包含 Top 30 網路特徵 + 系統指標）。

👻 [2. 幽靈欄位檢查]
   🟢 狀態: 完美！沒有殘留任何 Unnamed 欄位。

🎯 [3. 攻擊標籤檢查]
   - 標籤分佈: {0: 58257, 1: 19917}
   🟢 狀態: 完美！標籤已成功轉為數字 0 (正常) 與 1 (攻擊)。

🧼 [4. 資料品質檢查]
   🟢 狀態: 完美！全檔案無遺失值 (NaN)。
   🟢 狀態: 完美！所有特徵欄位皆為數值型態。
--------------------------------------------------
✅ 檢查完成！如果上方全是綠燈，你可以放心進入下一步：週期性處理 (No-Periodicity)。
